# Importamos las dependencias

#### Spark

In [1]:
# 1. Instalar Java (Spark lo necesita)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# 2. Descargar e instalar Spark (vamos a usar una versión específica compatible)
# Puedes verificar las últimas versiones en https://spark.apache.org/downloads.html
# Por ahora, usemos una versión estable conocida.
!wget -q https://archive.apache.org/dist/spark/spark-3.3.0/spark-3.3.0-bin-hadoop3.tgz
!tar xf spark-3.3.0-bin-hadoop3.tgz
!rm spark-3.3.0-bin-hadoop3.tgz # Limpiar el archivo tgz

# 3. Instalar PySpark y Findspark
!pip install -q findspark
!pip install -q pyspark==3.3.0 # Instalar la misma versión de pyspark

# 4. Configurar las variables de entorno
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.3.0-bin-hadoop3"

# 5. Inicializar findspark
import findspark
findspark.init()

# 6. Crear la SparkSession (esto ahora debería funcionar)
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Practica 4") \
    .getOrCreate()

print(spark)
print("¡SparkSession creada exitosamente!")

^C

gzip: stdin: unexpected end of file
tar: Unexpected EOF in archive
tar: Unexpected EOF in archive
tar: Error is not recoverable: exiting now
¡SparkSession creada exitosamente!


In [2]:
# Configuración para manejar errores de parseo de fecha en Spark 3.x
# Esto debe ejecutarse una vez por SparkSession, preferiblemente al inicio.
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
print("Configuración 'spark.sql.legacy.timeParserPolicy' establecida a 'LEGACY'.")

# Opcional: Para verificar si la configuración se aplicó
print(spark.conf.get("spark.sql.legacy.timeParserPolicy"))

Configuración 'spark.sql.legacy.timeParserPolicy' establecida a 'LEGACY'.
LEGACY


In [3]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MiLimpiezaDeDatos") \
    .master("local[*]") \
    .getOrCreate()

print(spark) # Esto debería mostrarte información sobre tu sesión de Spark

In [4]:
from pyspark.sql.types import StringType, ShortType, IntegerType, DoubleType, DateType
from pyspark.sql.functions import col, count, when

#### ML

In [5]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, TSNE
import seaborn as sns
import matplotlib.pyplot as plt

# Carga de datos

In [6]:
# Carga de datos

# Copiamos los datos de Google Drive al entorno local de Colab para una lectura más estable
ruta_drive = "/content/drive/MyDrive/Apuntes/Octavo semestre/Minería de datos/Programas/Práctica 4/Datos"
ruta_local = "/content/datos_practica4" # Directorio local donde copiaremos los archivos

# Formato real de tus fechas: dd/MM/yyyy
fecha_formato_real = "dd/MM/yyyy"

# Creamos el directorio local si no existe
!mkdir -p "{ruta_local}"

# Copiamos los archivos desde Google Drive al directorio local
# Usamos -r para copiar directorios recursivamente si ruta_drive apunta a una carpeta
# Usamos -v para ver el progreso (opcional)
!cp -r "{ruta_drive}"/* "{ruta_local}"/

# Ahora Spark leerá los datos desde la ruta local
df = spark.read.csv(ruta_local, header = True, inferSchema = True, dateFormat=fecha_formato_real)

In [7]:
df.show(5)

+--------------+------------+-------+---------------------+------------+-----------+--------------------+------------+-----------+
|Genero_Usuario|Edad_Usuario|   Bici|Ciclo_Estacion_Retiro|Fecha_Retiro|Hora_Retiro|Ciclo_EstacionArribo|Fecha_Arribo|Hora_Arribo|
+--------------+------------+-------+---------------------+------------+-----------+--------------------+------------+-----------+
|             M|          41|5254487|                  121|  31/08/2024|   23:55:44|                 134|  01/09/2024|   00:00:00|
|             M|          25|4809562|                  135|  31/08/2024|   23:52:38|                 576|  01/09/2024|   00:00:03|
|             M|          35|5842372|                  041|  31/08/2024|   23:56:21|                 010|  01/09/2024|   00:00:07|
|             O|          26|7207137|                  111|  31/08/2024|   23:43:50|                 136|  01/09/2024|   00:00:07|
|             M|          20|3196177|                  295|  31/08/2024|   23:55:43

# Ingeniería de datos

## Exploración de los datos

In [ ]:
# Veamos la cantidad de registros que tiene nuestro df ---
df.count()

120393618

In [ ]:
# --- Eliminamos las columnas que no nos interesan ---
df = df.drop(*['Bici', 'Hora_Retiro', 'Hora_Arribo'])
df.columns

['Genero_Usuario',
 'Edad_Usuario',
 'Ciclo_Estacion_Retiro',
 'Fecha_Retiro',
 'Ciclo_EstacionArribo',
 'Fecha_Arribo']

In [ ]:
# Y vemos la cantidad de nulos que tenemos
from pyspark.sql.functions import col

for columna in df.columns:
  num_nulos = df.filter(col(columna).isNull()).count()
  print(f"Columna '{columna}': ({num_nulos}, {(num_nulos/120393618)*100:.2f}%) nulos")

Columna 'Genero_Usuario': (463359, 0.38%) nulos
Columna 'Edad_Usuario': (11095, 0.01%) nulos
Columna 'Ciclo_Estacion_Retiro': (11095, 0.01%) nulos
Columna 'Fecha_Retiro': (11095, 0.01%) nulos
Columna 'Ciclo_EstacionArribo': (11084, 0.01%) nulos
Columna 'Fecha_Arribo': (11084, 0.01%) nulos


Ahora, vamos a cambiar el tipo de dato de las columnas

In [ ]:
# --- Vemos el Schema para saber cuales si debemos de cambiar ---
df.printSchema()

root
 |-- Genero_Usuario: string (nullable = true)
 |-- Edad_Usuario: string (nullable = true)
 |-- Ciclo_Estacion_Retiro: string (nullable = true)
 |-- Fecha_Retiro: string (nullable = true)
 |-- Ciclo_EstacionArribo: string (nullable = true)
 |-- Fecha_Arribo: string (nullable = true)



In [ ]:
# Por seguridad, copiamos el df en otra variable
df2 = df

In [ ]:
df2 = df2.withColumn("Edad_Usuario", col("Edad_Usuario").cast(ShortType()))

In [ ]:
# --- Vemos el Schema para confirmar los cambios ---
df2.printSchema()

root
 |-- Genero_Usuario: string (nullable = true)
 |-- Edad_Usuario: short (nullable = true)
 |-- Ciclo_Estacion_Retiro: string (nullable = true)
 |-- Fecha_Retiro: string (nullable = true)
 |-- Ciclo_EstacionArribo: string (nullable = true)
 |-- Fecha_Arribo: string (nullable = true)



In [ ]:
df2.show(5)

+--------------+------------+---------------------+------------+--------------------+------------+
|Genero_Usuario|Edad_Usuario|Ciclo_Estacion_Retiro|Fecha_Retiro|Ciclo_EstacionArribo|Fecha_Arribo|
+--------------+------------+---------------------+------------+--------------------+------------+
|             M|          41|                  121|  31/08/2024|                 134|  01/09/2024|
|             M|          25|                  135|  31/08/2024|                 576|  01/09/2024|
|             M|          35|                  041|  31/08/2024|                 010|  01/09/2024|
|             O|          26|                  111|  31/08/2024|                 136|  01/09/2024|
|             M|          20|                  295|  31/08/2024|                 298|  01/09/2024|
+--------------+------------+---------------------+------------+--------------------+------------+
only showing top 5 rows



In [ ]:
# --- Clasificación de las variables ---

# Después de lo que hicimos, podemos clasificar las variables de manera manual
varc = ['Edad_Usuario']
vard = vard = [
    "Genero_Usuario",
    "Ciclo_Estacion_Retiro",
    "Ciclo_EstacionArribo"
]

# Por si acaso, separamos las fechas en un grupo especial
var_fechas = [
    "Fecha_Retiro",
    "Fecha_Arribo"
]

## Tratamiento de los nulos y datos atípicos

Como pudimos ver unas celdas más arriba, la cantidad de nulos es porcentualmente super baja, así que los vamos a eliminar directamente.

In [ ]:
df2 = df2.dropna()

In [ ]:
df2.count()

119928870

Vemos que casí no disminuyó la cantidad de registros

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

print("### Tratamiento de Valores Extremos (Outliers) ###")

# Revisa y ajusta esta lista si es necesario.
# Si 'Edad_Usuario' es la única continua, se mantiene.
# Si tienes otras columnas que consideres continuas, añádelas aquí.
varc = ["Edad_Usuario"] # Asumo que 'Edad_Usuario' es tu única columna continua numérica.

print(f"\nVariables continuas a procesar para outliers: {varc}")
print("-" * 70)

# Asegúrate de que df2 sea tu DataFrame actual y limpio
# df2_processed = df2 # Ya que Spark DataFrames son inmutables, la asignación directa crea un nuevo DF con cada transformación

for var in varc:
    print(f"Procesando variable: {var}")

    # Es buena práctica asegurar que la columna es numérica para 'approxQuantile'
    # y filtrar nulos temporalmente para el cálculo de cuantiles.
    df_temp_for_quantiles = df2.select(col(var).cast(DoubleType()).alias(var)).dropna()

    # Si después de limpiar nulos, esta columna está vacía, no podemos calcular percentiles
    if df_temp_for_quantiles.count() == 0:
        print(f"  Advertencia: La columna '{var}' no tiene valores no nulos para calcular percentiles. Se omite el filtro de outliers.")
        continue # Saltar a la siguiente variable si no hay datos.

    # Calcular los percentiles 1% y 99%
    # El 0.001 es la tolerancia de error relativa para 'approxQuantile'
    quantiles = df_temp_for_quantiles.approxQuantile(var, [0.01, 0.99], 0.001)
    perc1 = quantiles[0]
    perc99 = quantiles[1]

    print(f"  - Percentil 1% para '{var}': {perc1:.3f}")
    print(f"  - Percentil 99% para '{var}': {perc99:.3f}")

    # Contar filas antes de aplicar el filtro para ver el impacto
    initial_count_for_var = df2.count()

    # Filtrar el DataFrame: mantener solo los valores dentro del rango de percentiles.
    # Esta operación elimina las filas que están fuera de ese rango.
    df2 = df2.filter(
        (col(var) >= perc1) & (col(var) <= perc99)
    )

    final_count_for_var = df2.count()
    print(f"  - Filas eliminadas para '{var}': {initial_count_for_var - final_count_for_var}")

print("-" * 70)
print(f"\nShape final del DataFrame Spark después de eliminar outliers: ({df2.count()}, {len(df2.columns)})")

print("\nEsquema del DataFrame Spark después de la eliminación de outliers:")
df2.printSchema()

print("\nPrimeras 5 filas del DataFrame Spark después de la eliminación de outliers:")
df2.show(5)

### Tratamiento de Valores Extremos (Outliers) ###

Variables continuas a procesar para outliers: ['Edad_Usuario']
----------------------------------------------------------------------
Procesando variable: Edad_Usuario
  - Percentil 1% para 'Edad_Usuario': 19.000
  - Percentil 99% para 'Edad_Usuario': 65.000
  - Filas eliminadas para 'Edad_Usuario': 1559694
----------------------------------------------------------------------

Shape final del DataFrame Spark después de eliminar outliers: (118369176, 6)

Esquema del DataFrame Spark después de la eliminación de outliers:
root
 |-- Genero_Usuario: string (nullable = true)
 |-- Edad_Usuario: short (nullable = true)
 |-- Ciclo_Estacion_Retiro: string (nullable = true)
 |-- Fecha_Retiro: string (nullable = true)
 |-- Ciclo_EstacionArribo: string (nullable = true)
 |-- Fecha_Arribo: string (nullable = true)


Primeras 5 filas del DataFrame Spark después de la eliminación de outliers:
+--------------+------------+---------------------+------

## Ingeniería de características

In [ ]:
from pyspark.sql.functions import col, to_date, year, month, dayofmonth, dayofweek, datediff
from pyspark.sql.types import DateType

print("### Corrección de Tipos de Datos para Fechas e Ingeniería de Características en df3 ###")

# Creamos df3 como una copia de df2 para trabajar sobre ella
df3 = df2 # Spark DataFrames son inmutables; esta es una "vista" pero asegura que los cambios son sobre una nueva referencia.

# Paso 1: Forzar la conversión de las columnas de fecha de String a DateType en df3
# El formato de tus fechas es "dd/MM/yyyy".
fecha_formato_original_str = "dd/MM/yyyy"

print(f"\nConvirtiendo 'Fecha_Retiro' y 'Fecha_Arribo' de String a DateType en df3 usando el formato: {fecha_formato_original_str}")

df3 = df3.withColumn("Fecha_Retiro", to_date(col("Fecha_Retiro"), fecha_formato_original_str).cast(DateType())) \
         .withColumn("Fecha_Arribo", to_date(col("Fecha_Arribo"), fecha_formato_original_str).cast(DateType()))

print("\nEsquema de df3 después de la conversión de tipos de fecha:")
df3.printSchema() # Confirma que ahora son DateType

# Paso 2: Ingeniería de Características - Extracción de Características de Fechas en df3
# Ahora que las fechas en df3 son de tipo 'date', las funciones de extracción funcionarán.

print("\n--- Extrayendo características de 'Fecha_Retiro' y 'Fecha_Arribo' en df3 ---")

# Extracción de características de Fecha_Retiro
if "Fecha_Retiro" in df3.columns and df3.schema["Fecha_Retiro"].dataType.typeName() == "date":
    df3 = df3.withColumn("Fecha_Retiro_Anio", year(col("Fecha_Retiro"))) \
             .withColumn("Fecha_Retiro_Mes", month(col("Fecha_Retiro"))) \
             .withColumn("Fecha_Retiro_DiaDelMes", dayofmonth(col("Fecha_Retiro"))) \
             .withColumn("Fecha_Retiro_DiaDeLaSemana", dayofweek(col("Fecha_Retiro")))
    print("  - Columnas de Fecha_Retiro añadidas correctamente a df3.")
else:
    print("  - ERROR: 'Fecha_Retiro' en df3 no es de tipo 'date' o no existe DESPUÉS de la conversión. Esto no debería pasar.")


# Extracción de características de Fecha_Arribo
if "Fecha_Arribo" in df3.columns and df3.schema["Fecha_Arribo"].dataType.typeName() == "date":
    df3 = df3.withColumn("Fecha_Arribo_Anio", year(col("Fecha_Arribo"))) \
             .withColumn("Fecha_Arribo_Mes", month(col("Fecha_Arribo"))) \
             .withColumn("Fecha_Arribo_DiaDelMes", dayofmonth(col("Fecha_Arribo"))) \
             .withColumn("Fecha_Arribo_DiaDeLaSemana", dayofweek(col("Fecha_Arribo")))
    print("  - Columnas de Fecha_Arribo añadidas correctamente a df3.")
else:
    print("  - ERROR: 'Fecha_Arribo' en df3 no es de tipo 'date' o no existe DESPUÉS de la conversión. Esto no debería pasar.")

# Calcular la duración en días entre Retiro y Arribo
if "Fecha_Retiro" in df3.columns and "Fecha_Arribo" in df3.columns and \
   df3.schema["Fecha_Retiro"].dataType.typeName() == "date" and df3.schema["Fecha_Arribo"].dataType.typeName() == "date":
    df3 = df3.withColumn("Duracion_Dias", datediff(col("Fecha_Arribo"), col("Fecha_Retiro")))
    print("  - Columna 'Duracion_Dias' añadida correctamente a df3.")
else:
    print("  - ERROR: No se pudo calcular 'Duracion_Dias' en df3. Ambas columnas de fecha no son de tipo 'date'.")


print("\nEsquema FINAL de df3 con nuevas características de fecha:")
df3.printSchema()

print("\nPrimeras 5 filas de df3 con nuevas características de fecha:")
df3.show(5)

### Corrección de Tipos de Datos para Fechas e Ingeniería de Características en df3 ###

Convirtiendo 'Fecha_Retiro' y 'Fecha_Arribo' de String a DateType en df3 usando el formato: dd/MM/yyyy

Esquema de df3 después de la conversión de tipos de fecha:
root
 |-- Genero_Usuario: string (nullable = true)
 |-- Edad_Usuario: short (nullable = true)
 |-- Ciclo_Estacion_Retiro: string (nullable = true)
 |-- Fecha_Retiro: date (nullable = true)
 |-- Ciclo_EstacionArribo: string (nullable = true)
 |-- Fecha_Arribo: date (nullable = true)


--- Extrayendo características de 'Fecha_Retiro' y 'Fecha_Arribo' en df3 ---
  - Columnas de Fecha_Retiro añadidas correctamente a df3.
  - Columnas de Fecha_Arribo añadidas correctamente a df3.
  - Columna 'Duracion_Dias' añadida correctamente a df3.

Esquema FINAL de df3 con nuevas características de fecha:
root
 |-- Genero_Usuario: string (nullable = true)
 |-- Edad_Usuario: short (nullable = true)
 |-- Ciclo_Estacion_Retiro: string (nullable = true)
 |-- 

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.sql.functions import col # Solo para asegurar que col esté importado

print("### Preparación de Datos para Modelación No Supervisada (SIN ESCALADO) ###")

# 1. Definir las listas finales de columnas para el modelo (manteniendo las que ya definiste)
numeric_features = [
    "Edad_Usuario",
    "Fecha_Retiro_Anio",
    "Fecha_Retiro_Mes",
    "Fecha_Retiro_DiaDelMes",
    "Fecha_Retiro_DiaDeLaSemana",
    "Fecha_Arribo_Anio",
    "Fecha_Arribo_Mes",
    "Fecha_Arribo_DiaDelMes",
    "Fecha_Arribo_DiaDeLaSemana",
    "Duracion_Dias"
]

categorical_features = [
    "Genero_Usuario",
    "Ciclo_Estacion_Retiro",
    "Ciclo_EstacionArribo"
]

print(f"\nCaracterísticas Numéricas/Continuas para el modelo: {numeric_features}")
print(f"Características Categóricas para el modelo: {categorical_features}")
print("-" * 70)


# 2. Codificación de Variables Categóricas (One-Hot Encoding)
indexed_features = [f"{col}_indexed" for col in categorical_features]
encoded_features = [f"{col}_encoded" for col in categorical_features]

indexers = [
    StringIndexer(inputCol=col_name, outputCol=f"{col_name}_indexed", handleInvalid="keep")
    for col_name in categorical_features
]

encoders = [
    OneHotEncoder(inputCol=indexed_col, outputCol=encoded_col)
    for indexed_col, encoded_col in zip(indexed_features, encoded_features)
]

pipeline_categorical = Pipeline(stages=indexers + encoders)

print("\nAplicando StringIndexer y OneHotEncoder a las características categóricas...")
df_encoded = pipeline_categorical.fit(df3).transform(df3) # Usamos df3 como entrada

print("Esquema de df3 después de la codificación categórica:")
df_encoded.printSchema()
print(f"Shape: ({df_encoded.count()}, {len(df_encoded.columns)})")
df_encoded.show(5)


# 3. Ensamblar todas las características en un solo vector
all_features_for_vector_assembler = numeric_features + encoded_features

# CAMBIO IMPORTANTE: handleInvalid="keep" en VectorAssembler
# Esto maneja los nulos que pueda haber en las columnas a ensamblar, reemplazándolos con 0
# y añadiendo una columna indicadora de nulo.
assembler = VectorAssembler(inputCols=all_features_for_vector_assembler, outputCol="features_raw", handleInvalid="keep")

print("\nEnsamblando todas las características en un solo vector 'features_raw' (con manejo de nulos)...")
df_vectorized = assembler.transform(df_encoded)

print("Esquema de df3 después de ensamblar las características:")
df_vectorized.printSchema()
df_vectorized.select("features_raw").show(5, truncate=False)


# *** SE OMITE EL ESCALADO (StandardScaler)  ***
# df_scaled = df_vectorized # Ahora, df_vectorized es tu DataFrame final listo para la modelación

print("\n*** El escalado de características ha sido omitido. ***")
print("\nTu DataFrame final listo para modelación es df_vectorized (contiene la columna 'features_raw').")
df_final_for_model = df_vectorized # Renombramos para mayor claridad.

print("\nEsquema del DataFrame final para modelación:")
df_final_for_model.printSchema()
df_final_for_model.select("features_raw").show(5, truncate=False)

### Preparación de Datos para Modelación No Supervisada (SIN ESCALADO) ###

Características Numéricas/Continuas para el modelo: ['Edad_Usuario', 'Fecha_Retiro_Anio', 'Fecha_Retiro_Mes', 'Fecha_Retiro_DiaDelMes', 'Fecha_Retiro_DiaDeLaSemana', 'Fecha_Arribo_Anio', 'Fecha_Arribo_Mes', 'Fecha_Arribo_DiaDelMes', 'Fecha_Arribo_DiaDeLaSemana', 'Duracion_Dias']
Características Categóricas para el modelo: ['Genero_Usuario', 'Ciclo_Estacion_Retiro', 'Ciclo_EstacionArribo']
----------------------------------------------------------------------

Aplicando StringIndexer y OneHotEncoder a las características categóricas...
Esquema de df3 después de la codificación categórica:
root
 |-- Genero_Usuario: string (nullable = true)
 |-- Edad_Usuario: short (nullable = true)
 |-- Ciclo_Estacion_Retiro: string (nullable = true)
 |-- Fecha_Retiro: date (nullable = true)
 |-- Ciclo_EstacionArribo: string (nullable = true)
 |-- Fecha_Arribo: date (nullable = true)
 |-- Fecha_Retiro_Anio: integer (nullable = tr

# Modelado

In [ ]:
from pyspark.ml.feature import PCA
from pyspark.sql.functions import col
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns # Para visualización

print("### Modelación No Supervisada - PCA (Análisis de Componentes Principales) ###")

# Aplicar PCA
# k es el número de componentes principales que quieres obtener.
pca = PCA(k=3, inputCol="features_raw", outputCol="pca_features")

print("\nAplicando PCA a las características (features_raw)...")
pca_model = pca.fit(df_final_for_model) # Entrena el modelo PCA con tus datos preparados

# Transforma el DataFrame para obtener las nuevas componentes principales
df_pca_result = pca_model.transform(df_final_for_model)

print("\nEsquema de df_pca_result con las nuevas componentes principales:")
df_pca_result.printSchema()
df_pca_result.select("features_raw", "pca_features").show(5, truncate=False)

# Acceder a la varianza explicada por cada componente (similar a explained_variance_ratio_ en sklearn)
explained_variance = pca_model.explainedVariance.toArray()
print(f"\nVarianza explicada por cada componente principal: {explained_variance}")
print(f"Varianza explicada acumulada: {explained_variance.cumsum()}")

# Visualizar la varianza explicada (opcional, si quieres ver el "elbow method" para PCA)
# Cuidado: Si el DataFrame es muy grande, .toPandas() puede consumir mucha RAM.
# Para graficar, solo necesitamos los valores de varianza.
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(explained_variance) + 1), explained_variance.cumsum(), marker='o', linestyle='--')
plt.title('Varianza Acumulada Explicada por Componentes Principales')
plt.xlabel('Número de Componentes Principales')
plt.ylabel('Varianza Explicada Acumulada')
plt.grid(True)
plt.show()

# Para visualización 3D (requiere recolectar a Pandas y puede ser lento/memoria)
# Si df_pca_result es muy grande, considera muestrear: .sample(0.01) antes de .toPandas()
# O solo tomar las columnas necesarias para la visualización 3D.
print("\nPreparando datos para visualización 3D (esto puede tardar y consumir RAM si el dataset es grande)...")

# Seleccionar solo las columnas necesarias para la visualización 3D y convertir a Pandas
# Puedes muestrear si la RAM de Colab es limitada: .sample(False, 0.01, seed=42)
df_pca_for_plot_pandas = df_pca_result.select("pca_features").limit(10000).toPandas() # Limitar a 10,000 filas para evitar agotamiento de RAM

# Convertir la columna 'pca_features' (Vector) a columnas separadas en Pandas
df_pca_for_plot_pandas[['p1', 'p2', 'p3']] = df_pca_for_plot_pandas['pca_features'].apply(lambda x: pd.Series(x.toArray()))

import plotly.express as px
fig = px.scatter_3d(df_pca_for_plot_pandas, x='p1', y='p2', z='p3',
              title='PCA 3D de Características (Muestra)')
fig.show()

print("\nPCA aplicado exitosamente. Las componentes principales están en 'df_pca_result'.")

### Modelación No Supervisada - PCA (Análisis de Componentes Principales) ###

Aplicando PCA a las características (features_raw)...


Py4JJavaError: An error occurred while calling o1953.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 67 in stage 186.0 failed 1 times, most recent failure: Lost task 67.0 in stage 186.0 (TID 4855) (3192dc8eca6b executor driver): org.apache.spark.SparkUpgradeException: You may get a different result due to the upgrading to Spark >= 3.0: Fail to parse '30/08/22' in the new parser. You can set spark.sql.legacy.timeParserPolicy to LEGACY to restore the behavior before Spark 3.0, or set to CORRECTED and treat it as an invalid datetime string.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failToParseDateTimeInNewParserError(QueryExecutionErrors.scala:1066)
	at org.apache.spark.sql.catalyst.util.DateTimeFormatterHelper$$anonfun$checkParsedDiff$1.applyOrElse(DateTimeFormatterHelper.scala:148)
	at org.apache.spark.sql.catalyst.util.DateTimeFormatterHelper$$anonfun$checkParsedDiff$1.applyOrElse(DateTimeFormatterHelper.scala:141)
	at scala.runtime.AbstractPa